In [237]:
from numba import njit, types, prange
from numba.typed import List, Dict
from numpy import ndarray as CPUArray

import numpy as np

In [238]:
list(np.ndenumerate(np.random.randn(3, 3)))

[((0, 0), np.float64(0.7827188926363816)),
 ((0, 1), np.float64(0.9029287028480432)),
 ((0, 2), np.float64(0.8830227058990934)),
 ((1, 0), np.float64(1.6618618277178359)),
 ((1, 1), np.float64(1.9610152209294744)),
 ((1, 2), np.float64(1.2177666020066706)),
 ((2, 0), np.float64(1.7288611091768502)),
 ((2, 1), np.float64(0.2317600888727438)),
 ((2, 2), np.float64(-0.5913964644254673))]

In [239]:
@njit
def get_enumeration(array: CPUArray):
    positions: list[tuple[int, ...]] = [index for index, value in np.ndenumerate(array)]
    count = len(positions)
    return positions, count

In [240]:
res0 = get_enumeration(np.random.randn(3, 3))
res0[0][:3], res0[1]

([(0, 0), (0, 1), (0, 2)], 9)

In [241]:
res0 = get_enumeration(np.random.randn(3, 3, 3))
res0[0][:3], res0[1]

([(0, 0, 0), (0, 0, 1), (0, 0, 2)], 27)

In [242]:
def _get_enumeration(array: CPUArray):
    positions = [index for index, value in np.ndenumerate(array)]
    count = len(positions)
    return positions, count

In [243]:
%%timeit -r 10 -n 10
get_enumeration(np.random.randn(100, 100, 100));

192 ms ± 5.85 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [244]:
%%timeit -r 10 -n 10
_get_enumeration(np.random.randn(100, 100, 100));

515 ms ± 32.6 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [273]:
@njit(nogil=True)
def process_sum(array: CPUArray):
    indices, total = get_enumeration(array)
    summation = 0.0
    for i in prange(total):
        summation += array[indices[i]]
    return summation

In [274]:
process_sum(np.random.randn(3, 3))

1.3689807127137992

In [275]:
process_sum(np.random.randn(10, 10, 10))

-25.930997484911813

In [276]:
def _process_sum(array: CPUArray):
    indices, total = _get_enumeration(array)
    summation = 0.0
    for i in prange(total):
        summation += array[indices[i]]
    return summation

In [277]:
%%timeit -r 10 -n 3
process_sum(np.random.randn(100, 128, 128));

96.2 ms ± 7 ms per loop (mean ± std. dev. of 10 runs, 3 loops each)


In [279]:
%%timeit -r 10 -n 3
_process_sum(np.random.randn(100, 128, 128));

421 ms ± 11.7 ms per loop (mean ± std. dev. of 10 runs, 3 loops each)
